<a href="https://colab.research.google.com/github/ArjunBhakta/Data-Science-Cohort-20/blob/main/Chinook_100_Questions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SQLite – Chinook Database
sqlite3 is part of the Python standard library — no installation needed.

In [1]:
import sqlite3 as db
import pandas as pd
import urllib.request
import os

In [2]:
# Download Chinook only if not already present
# Using GitHub source — the tutorial site blocks programmatic downloads
DB_FILE = "chinook.db"
DB_URL  = "https://github.com/lerocha/chinook-database/raw/master/ChinookDatabase/DataSources/Chinook_Sqlite.sqlite"

if not os.path.exists(DB_FILE):
    print("Downloading chinook.db ...")
    req = urllib.request.Request(DB_URL, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req) as resp, open(DB_FILE, "wb") as f:
        f.write(resp.read())
    print("Done.")
else:
    print(f"{DB_FILE} already exists — skipping download.")

Done.


In [3]:
# Connect and inspect tables
conn = db.connect(DB_FILE)

tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;", conn)
tables

,name
0,Album
1,Artist
2,Customer
3,Employee
4,Genre
5,Invoice
6,InvoiceLine
7,MediaType
8,Playlist
9,PlaylistTrack


---
# Chinook SQL Exercises
## Schema Reference
| Table | Key Columns |
|---|---|
| **Artist** | ArtistId, Name |
| **Album** | AlbumId, Title, ArtistId |
| **Track** | TrackId, Name, AlbumId, MediaTypeId, GenreId, Composer, Milliseconds, Bytes, UnitPrice |
| **Genre** | GenreId, Name |
| **MediaType** | MediaTypeId, Name |
| **Playlist** | PlaylistId, Name |
| **PlaylistTrack** | PlaylistId, TrackId |
| **Employee** | EmployeeId, LastName, FirstName, Title, ReportsTo, BirthDate, HireDate, Address, City, State, Country |
| **Customer** | CustomerId, FirstName, LastName, Company, City, State, Country, Email, SupportRepId |
| **Invoice** | InvoiceId, CustomerId, InvoiceDate, BillingCountry, Total |
| **InvoiceLine** | InvoiceLineId, InvoiceId, TrackId, UnitPrice, Quantity |

---
## Level 1 — Basic SELECT & Filtering (Q1–15)
Single-table queries, simple WHERE clauses, no joins.

**Q1.** List all artists.

In [4]:
pd.read_sql("""
    SELECT name FROM Artist

""", conn)

,Name
0,AC/DC
1,Accept
2,Aerosmith
3,Alanis Morissette
4,Alice In Chains
...,...
270,"Mela Tenenbaum, Pro Musica Prague & Richard Kapp"
271,Emerson String Quartet
272,"C. Monteverdi, Nigel Rogers - Chiaroscuro; Lon..."
273,Nash Ensemble


**Q2.** List all genres.

In [5]:
pd.read_sql("""
    SELECT name AS Genre FROM Genre

""", conn)

,Genre
0,Rock
1,Jazz
2,Metal
3,Alternative & Punk
4,Rock And Roll
5,Blues
6,Latin
7,Reggae
8,Pop
9,Soundtrack


**Q3.** List all media types.

In [6]:
pd.read_sql("""
    SELECT Name AS media From MediaType
""", conn)

,media
0,MPEG audio file
1,Protected AAC audio file
2,Protected MPEG-4 video file
3,Purchased AAC audio file
4,AAC audio file


**Q4.** How many tracks are in the database?

In [7]:
pd.read_sql("""
SELECT COUNT(*) FROM  Track
""", conn)

,COUNT(*)
0,3503


**Q5.** How many customers are there?

In [8]:
pd.read_sql("""
    SELECT Count(*) FROM Customer

""", conn)

,Count(*)
0,59


**Q6.** List all customers from the USA.

In [9]:
pd.read_sql("""
    SELECT CustomerID, FirstName, LastName, Country
    FROM Customer
    WHERE Country = 'USA'

""", conn)

,CustomerId,FirstName,LastName,Country
0,16,Frank,Harris,USA
1,17,Jack,Smith,USA
2,18,Michelle,Brooks,USA
3,19,Tim,Goyer,USA
4,20,Dan,Miller,USA
5,21,Kathy,Chase,USA
6,22,Heather,Leacock,USA
7,23,John,Gordon,USA
8,24,Frank,Ralston,USA
9,25,Victor,Stevens,USA


**Q7.** List all unique countries that customers come from.

In [10]:
pd.read_sql("""
    SELECT DISTINCT Country
    FROM Customer

""", conn)

,Country
0,Brazil
1,Germany
2,Canada
3,Norway
4,Czech Republic
5,Austria
6,Belgium
7,Denmark
8,USA
9,Portugal


**Q8.** Show tracks where the UnitPrice is greater than $0.99.

In [11]:
pd.read_sql("""
    Select TrackID, UnitPrice, Name
    FROM Track
    WHERE UnitPrice > 0.99

""", conn)

,TrackId,UnitPrice,Name
0,2819,1.99,Battlestar Galactica: The Story So Far
1,2820,1.99,Occupation / Precipice
2,2821,1.99,"Exodus, Pt. 1"
3,2822,1.99,"Exodus, Pt. 2"
4,2823,1.99,Collaborators
...,...,...,...
208,3362,1.99,"There's No Place Like Home, Pt. 1"
209,3363,1.99,"There's No Place Like Home, Pt. 2"
210,3364,1.99,"There's No Place Like Home, Pt. 3"
211,3428,1.99,Branch Closing


**Q9.** List all tracks longer than 5 minutes (300,000 milliseconds), showing name and length.

In [12]:
pd.read_sql("""
    SELECT TRACKID, Name, Milliseconds
    FROM Track
    WHERE Milliseconds > 300000

""", conn)

,TrackId,Name,Milliseconds
0,1,For Those About To Rock (We Salute You),343719
1,2,Balls to the Wall,342562
2,5,Princess of the Dawn,375418
3,15,Go Down,331180
4,17,Let There Be Rock,366654
...,...,...,...
1064,3486,"Act IV, Symphony",364296
1065,3487,"3 Gymnopédies: No.1 - Lent Et Grave, No.3 - Le...",385506
1066,3489,Symphony No. 2: III. Allegro vivace,376510
1067,3493,"Metopes, Op. 29: Calypso",333669


**Q10.** List all tracks sorted by length, longest first. Show name and Milliseconds.

In [13]:
pd.read_sql("""
    SELECT Name, Milliseconds
    FROM Track
    ORDER BY Milliseconds DESC

""", conn)

,Name,Milliseconds
0,Occupation / Precipice,5286953
1,Through a Looking Glass,5088838
2,"Greetings from Earth, Pt. 1",2960293
3,The Man With Nine Lives,2956998
4,"Battlestar Galactica, Pt. 2",2956081
...,...,...
3498,Commercial 1,7941
3499,Oprah,6635
3500,A Statistic,6373
3501,Now Sports,4884


**Q11.** Find all customers where the Company field is NULL.

In [14]:
pd.read_sql("""
    SELECT CustomerId, FirstName, LastName, company
    FROM Customer
    WHERE company IS NOT NULL
""", conn)

,CustomerId,FirstName,LastName,Company
0,1,Luís,Gonçalves,Embraer - Empresa Brasileira de Aeronáutica S.A.
1,5,František,Wichterlová,JetBrains s.r.o.
2,10,Eduardo,Martins,Woodstock Discos
3,11,Alexandre,Rocha,Banco do Brasil S.A.
4,12,Roberto,Almeida,Riotur
5,14,Mark,Philips,Telus
6,15,Jennifer,Peterson,Rogers Canada
7,16,Frank,Harris,Google Inc.
8,17,Jack,Smith,Microsoft Corporation
9,19,Tim,Goyer,Apple Inc.


**Q12.** Find all tracks that have no Composer listed.

In [15]:
pd.read_sql("""
    SELECT name , TrackId , composer
    FROM Track
    Where composer IS NULL
""", conn)

,Name,TrackId,Composer
0,Desafinado,63,None
1,Garota De Ipanema,64,None
2,Samba De Uma Nota Só (One Note Samba),65,None
3,Por Causa De Você,66,None
4,Ligia,67,None
...,...,...,...
972,Slowness,3478,None
973,"A Midsummer Night's Dream, Op.61 Incidental Mu...",3481,None
974,"Étude 1, In C Major - Preludio (Presto) - Liszt",3496,None
975,"Erlkonig, D.328",3497,None


**Q13.** Find all tracks whose name contains the word 'Love' (case-insensitive).

In [16]:
pd.read_sql("""
    SELECT name
    From Track
    Where name Like '%love%'

""", conn)

,Name
0,Love In An Elevator
1,"Love, Hate, Love"
2,Let Me Love You Baby
3,My Love
4,The Girl I Love She Got Long Black Wavy Hair
...,...
109,Love Comes
110,Arms Around Your Love
111,Love Is a Losing Game
112,I Heard Love Is Blind


**Q14.** List all employees sorted by HireDate (earliest first). Show first name, last name, and title.

In [17]:
pd.read_sql("""
    SELECT EmployeeId, FirstName, LastName, Title, HireDate
    From Employee
    ORDER BY HireDate DESC

""", conn)

,EmployeeId,FirstName,LastName,Title,HireDate
0,8,Laura,Callahan,IT Staff,2004-03-04 00:00:00
1,7,Robert,King,IT Staff,2004-01-02 00:00:00
2,5,Steve,Johnson,Sales Support Agent,2003-10-17 00:00:00
3,6,Michael,Mitchell,IT Manager,2003-10-17 00:00:00
4,4,Margaret,Park,Sales Support Agent,2003-05-03 00:00:00
5,1,Andrew,Adams,General Manager,2002-08-14 00:00:00
6,2,Nancy,Edwards,Sales Manager,2002-05-01 00:00:00
7,3,Jane,Peacock,Sales Support Agent,2002-04-01 00:00:00


**Q15.** List the top 10 most expensive invoices (by Total), showing InvoiceId, CustomerId, and Total.

In [18]:
pd.read_sql("""
    SELECT InvoiceId, CustomerId, InvoiceDate, Total
    FROM Invoice
    ORDER BY Total DESC
    LIMIT 10

""", conn)

,InvoiceId,CustomerId,InvoiceDate,Total
0,404,6,2025-11-13 00:00:00,25.86
1,299,26,2024-08-05 00:00:00,23.86
2,96,45,2022-02-18 00:00:00,21.86
3,194,46,2023-04-28 00:00:00,21.86
4,89,7,2022-01-18 00:00:00,18.86
5,201,25,2023-05-29 00:00:00,18.86
6,88,57,2022-01-13 00:00:00,17.91
7,306,5,2024-09-05 00:00:00,16.86
8,313,43,2024-10-06 00:00:00,16.86
9,103,24,2022-03-21 00:00:00,15.86


---
## Level 2 — Aggregation & Grouping (Q16–30)
COUNT, SUM, AVG, MIN, MAX, GROUP BY, HAVING.

**Q15b.** Find all tracks in the 'Rock' genre **AND** with a UnitPrice greater than $0.99. (Practice: AND)

In [19]:
pd.read_sql("""
    SELECT t.Name ,t.UnitPrice, g.Name AS Genre
    FROM Track t
    JOIN Genre g ON t.GenreId = g.GenreId
    WHERE g.Name = 'Rock'
    AND t.UnitPrice > 0.99

""", conn)

# No results for this query, but it runs correctly

,Name,UnitPrice,Genre


**Q15c.** Find all customers from Canada **OR** Brazil. (Practice: OR)

In [20]:
pd.read_sql("""
    SELECT c.FirstName , c.LastName, c.Country
    FROM Customer c
    WHERE country = 'Brazil' or country = 'Canada'

""", conn)

,FirstName,LastName,Country
0,Luís,Gonçalves,Brazil
1,François,Tremblay,Canada
2,Eduardo,Martins,Brazil
3,Alexandre,Rocha,Brazil
4,Roberto,Almeida,Brazil
5,Fernanda,Ramos,Brazil
6,Mark,Philips,Canada
7,Jennifer,Peterson,Canada
8,Robert,Brown,Canada
9,Edward,Francis,Canada


**Q15d.** Find all invoices with a Total **BETWEEN** $5.00 and $15.00. Show InvoiceId, CustomerId, and Total. (Practice: BETWEEN)

In [21]:
pd.read_sql("""
    SELECT INVOICEID, CustomerId, Total
    From Invoice
    WHERE Total BETWEEN 5 and 15

""", conn)

,InvoiceId,CustomerId,Total
0,3,8,5.94
1,4,14,8.91
2,5,23,13.86
3,10,46,5.94
4,11,52,8.91
...,...,...,...
163,402,50,5.94
164,403,56,8.91
165,409,29,5.94
166,410,35,8.91


**Q15e.** Find all customers whose last name matches the pattern `'B_own'` (i.e. exactly 5 letters, starts with B, ends with own). (Practice: LIKE with `_` wildcard)

In [22]:
pd.read_sql("""
    SELECT LastName
    FROM Customer
    WHERE LastName LIKE 'B_own'
""", conn)

,LastName
0,Brown


**Q16.** How many tracks are in each genre? Show genre name and track count, sorted by count descending.

In [23]:
pd.read_sql("""
    SELECT g.name AS Genre, COUNT(*) AS TrackCount
    FROM Genre g
    JOIN Track t ON g.GenreId = t.GenreId
    GROUP BY g.name
    ORDER BY TrackCount DESC

""", conn)

,Genre,TrackCount
0,Rock,1297
1,Latin,579
2,Metal,374
3,Alternative & Punk,332
4,Jazz,130
5,TV Shows,93
6,Blues,81
7,Classical,74
8,Drama,64
9,R&B/Soul,61


**Q17.** How many albums does each artist have? Show artist name and album count.

In [24]:
pd.read_sql("""
    SELECT Artist.Name AS Artist, COUNT(*) AS AlbumCount
    FROM Artist
    JOIN ALBUM ON Artist.ArtistId = ALBUM.ArtistId
    GROUP BY Artist.Name
    ORDER BY AlbumCount DESC

""", conn)

,Artist,AlbumCount
0,Iron Maiden,21
1,Led Zeppelin,14
2,Deep Purple,11
3,U2,10
4,Metallica,10
...,...,...
199,"Academy of St. Martin in the Fields, John Birc...",1
200,Academy of St. Martin in the Fields Chamber En...,1
201,Academy of St. Martin in the Fields & Sir Nevi...,1
202,Aaron Goldberg,1


**Q18.** What is the total revenue from all invoices?

In [25]:
pd.read_sql("""
    SELECT Sum(total) as TotalRevenue
    FROM Invoice
""", conn)

,TotalRevenue
0,2328.6


**Q19.** What is the average invoice total?

In [26]:
pd.read_sql("""
    SELECT AVG(Total) as AverageInvoiceTotal
    FROM Invoice
""", conn)

,AverageInvoiceTotal
0,5.651942


**Q20.** How many customers are in each country? Sort by count descending.

In [27]:
pd.read_sql("""
    SELECT Country, Count(*) AS CountryCount
    FROM Customer
    GROUP BY Country
    ORDER BY CountryCount DESC
""", conn)

,Country,CountryCount
0,USA,13
1,Canada,8
2,France,5
3,Brazil,5
4,Germany,4
5,United Kingdom,3
6,Portugal,2
7,India,2
8,Czech Republic,2
9,Sweden,1


**Q21.** What is the total revenue per billing country? Sort by revenue descending.

In [28]:
pd.read_sql("""
    SELECT BillingCountry, SUM(Total) AS TotalRevenue
    FROM Invoice
    GROUP BY BillingCountry
    ORDER BY TotalRevenue DESC

""", conn)

,BillingCountry,TotalRevenue
0,USA,523.06
1,Canada,303.96
2,France,195.10
3,Brazil,190.10
4,Germany,156.48
5,United Kingdom,112.86
6,Czech Republic,90.24
7,Portugal,77.24
8,India,75.26
9,Chile,46.62


**Q22.** How many tracks does each album have? Show album title and track count.`

In [29]:
pd.read_sql("""
    SELECT Album.title AS Album, COUNT(Track.TrackId) AS TrackCount
    FROM Album
    JOIN Track ON Album.AlbumId = Track.AlbumId
    GROUP BY Album.title
    ORDER BY TrackCount DESC

""", conn)

,Album,TrackCount
0,Greatest Hits,57
1,Minha Historia,34
2,Unplugged,30
3,"Lost, Season 3",26
4,"The Office, Season 3",25
...,...,...
342,Allegri: Miserere,1
343,Adorate Deum: Gregorian Chant from the Proper ...,1
344,"Adams, John: The Chairman Dances",1
345,A Soprano Inspired,1


**Q23.** What is the maximum and minimum invoice total?

In [30]:
pd.read_sql("""
    SELECT Max(Total) AS MaxInvoiceTotal, Min(Total) AS MinInvoiceTotal, AVG(Total) AS AverageInvoiceTotal
    FROM Invoice

""", conn)

,MaxInvoiceTotal,MinInvoiceTotal,AverageInvoiceTotal
0,25.86,0.99,5.651942


**Q24.** How many tracks are there per media type?

In [31]:
pd.read_sql("""
    SELECT Count(*) AS TotalTracks, Track.MediaTypeId, MediaType.Name AS Media
    FROM Track
    JOIN MediaType ON Track.MediaTypeId = MediaType.MediaTypeId
    GROUP BY Track.MediaTypeId, MediaType.Name
""", conn)

,TotalTracks,MediaTypeId,Media
0,3034,1,MPEG audio file
1,237,2,Protected AAC audio file
2,214,3,Protected MPEG-4 video file
3,7,4,Purchased AAC audio file
4,11,5,AAC audio file


**Q25.** What is the total number of invoices per year?

In [32]:
pd.read_sql("""
    SELECT Count(*) AS InvoiceCount, strftime('%Y', invoiceDate) AS InvoiceYear
    FROM Invoice
    Group BY InvoiceYear

""", conn)

,InvoiceCount,InvoiceYear
0,83,2021
1,83,2022
2,83,2023
3,83,2024
4,80,2025


**Q26.** How many invoices does each customer have? Show customer ID and invoice count.

In [33]:
pd.read_sql("""
    SELECT Customer.CustomerId, Customer.FirstName, Customer.LastName, COUNT(Invoice.InvoiceId) AS TotalInvoices
    FROM Customer
    JOIN Invoice ON Customer.CustomerId = Invoice.CustomerId
    GROUP BY Customer.CustomerId, Customer.FirstName, Customer.LastName
""", conn)

,CustomerId,FirstName,LastName,TotalInvoices
0,1,Luís,Gonçalves,7
1,2,Leonie,Köhler,7
2,3,François,Tremblay,7
3,4,Bjørn,Hansen,7
4,5,František,Wichterlová,7
5,6,Helena,Holý,7
6,7,Astrid,Gruber,7
7,8,Daan,Peeters,7
8,9,Kara,Nielsen,7
9,10,Eduardo,Martins,7


**Q27.** What is the total length (in minutes) of all tracks combined?

In [34]:
pd.read_sql("""
    SELECT SUM(Milliseconds) / 60000.0 AS TotalMinutes
    FROM Track

""", conn)

,TotalMinutes
0,22979.634


**Q28.** Find genres that have more than 100 tracks.

In [35]:
pd.read_sql("""
    SELECT GENRE.Name AS GenreType, COUNT (*) AS TrackCount
    FROM Genre
    JOIN Track ON Genre.GenreId = Track.GenreId
    GROUP BY Genre.Name
    HAVING TrackCount > 100

""", conn)

,GenreType,TrackCount
0,Alternative & Punk,332
1,Jazz,130
2,Latin,579
3,Metal,374
4,Rock,1297


**Q29.** Find albums with more than 15 tracks.

In [36]:
pd.read_sql("""
    SELECT Album.Title AS ALBUM , Count(*) AS TrackCount
    FROM Album
    JOIN Track ON Album.AlbumId = Track.AlbumId
    GROUP BY Album.Title
    HAVING TrackCount > 15
    ORDER BY TrackCount DESC

""", conn)

,ALBUM,TrackCount
0,Greatest Hits,57
1,Minha Historia,34
2,Unplugged,30
3,"Lost, Season 3",26
4,"The Office, Season 3",25
...,...,...
56,Live On Two Legs [Live],16
57,Judas 0: B-Sides and Rarities,16
58,Garage Inc. (Disc 2),16
59,By The Way,16


**Q30.** What is the average track length in seconds, per genre?

In [37]:
pd.read_sql("""
    SELECT AVG(Milliseconds) / 1000.0 AS AverageTrackLengthSeconds, Genre.Name AS Genre
    FROM Track
    JOIN Genre ON Track.GenreId = Genre.GenreId
    GROUP BY Genre.Name
    ORDER BY AverageTrackLengthSeconds DESC

""", conn)

,AverageTrackLengthSeconds,Genre
0,2911.783038,Sci Fi & Fantasy
1,2625.549077,Science Fiction
2,2575.283781,Drama
3,2145.041022,TV Shows
4,1585.263706,Comedy
5,309.749444,Metal
6,302.985800,Electronica/Dance
7,297.452929,Heavy Metal
8,293.867568,Classical
9,291.755377,Jazz


---
## Level 3 — JOINs (Q31–50)
Combining multiple tables with INNER JOIN, LEFT JOIN.

**Q31.** List all tracks with their genre name.

In [38]:
pd.read_sql("""
    SELECT Track.Name AS Track, Genre.Name AS Genre
    FROM Track
    INNER JOIN Genre ON Track.GenreId = Genre.GenreId

""", conn)

,Track,Genre
0,For Those About To Rock (We Salute You),Rock
1,Balls to the Wall,Rock
2,Fast As a Shark,Rock
3,Restless and Wild,Rock
4,Princess of the Dawn,Rock
...,...,...
3498,Pini Di Roma (Pinien Von Rom) \ I Pini Della V...,Classical
3499,"String Quartet No. 12 in C Minor, D. 703 ""Quar...",Classical
3500,"L'orfeo, Act 3, Sinfonia (Orchestra)",Classical
3501,"Quintet for Horn, Violin, 2 Violas, and Cello ...",Classical


**Q32.** List all tracks with their album title and artist name.

In [39]:
pd.read_sql("""
    SELECT ALBUM.Title AS Album, Artist.Name AS Artist, Track.Name AS Track
    FROM Track
    LEFT JOIN Album ON Track.AlbumId = Album.AlbumId
    LEFT JOIN Artist ON Album.ArtistId = Artist.ArtistId

""", conn)

,Album,Artist,Track
0,For Those About To Rock We Salute You,AC/DC,For Those About To Rock (We Salute You)
1,Balls to the Wall,Accept,Balls to the Wall
2,Restless and Wild,Accept,Fast As a Shark
3,Restless and Wild,Accept,Restless and Wild
4,Restless and Wild,Accept,Princess of the Dawn
...,...,...,...
3498,Respighi:Pines of Rome,Eugene Ormandy,Pini Di Roma (Pinien Von Rom) \ I Pini Della V...
3499,Schubert: The Late String Quartets & String Qu...,Emerson String Quartet,"String Quartet No. 12 in C Minor, D. 703 ""Quar..."
3500,Monteverdi: L'Orfeo,"C. Monteverdi, Nigel Rogers - Chiaroscuro; Lon...","L'orfeo, Act 3, Sinfonia (Orchestra)"
3501,Mozart: Chamber Music,Nash Ensemble,"Quintet for Horn, Violin, 2 Violas, and Cello ..."


**Q33.** Show each invoice with the customer's full name (FirstName + LastName).

In [40]:
pd.read_sql("""
    Select Customer.FirstName || ' ' || Customer.LastName AS Name, Invoice.InvoiceId
    FROM Invoice
    LEFT JOIN Customer ON Invoice.CustomerId = Customer.CustomerId

""", conn)

,Name,InvoiceId
0,Luís Gonçalves,98
1,Luís Gonçalves,121
2,Luís Gonçalves,143
3,Luís Gonçalves,195
4,Luís Gonçalves,316
...,...,...
407,Puja Srivastava,45
408,Puja Srivastava,97
409,Puja Srivastava,218
410,Puja Srivastava,229


**Q34.** List all albums by AC/DC.

In [41]:
pd.read_sql("""
    SELECT ALBUM.Title AS Album, Artist.Name AS Artist
    FROM ALbum
    JOIN ARTIST ON Album.ArtistId = Artist.ArtistId
    WHERE Artist.Name = 'AC/DC'

""", conn)

,Album,Artist
0,For Those About To Rock We Salute You,AC/DC
1,Let There Be Rock,AC/DC


**Q35.** List all tracks in the 'Music' playlist.

In [42]:
pd.read_sql("""
    SELECT  Track.name as Track, Playlist.PlaylistId, Playlist.Name
    FROM Playlist
    Join PlaylistTrack ON Playlist.PlaylistId = PlaylistTrack.PlaylistId
    Join Track ON PlaylistTrack.TrackId = Track.TrackId
    WHERE Playlist.Name = 'Music'
""", conn)

,Track,PlaylistId,Name
0,"Band Members Discuss Tracks from ""Revelations""",1,Music
1,Revelations,1,Music
2,One and the Same,1,Music
3,Sound of a Gun,1,Music
4,Until We Fall,1,Music
...,...,...,...
6575,"A Midsummer Night's Dream, Op.61 Incidental Mu...",8,Music
6576,Koyaanisqatsi,8,Music
6577,"Act IV, Symphony",8,Music
6578,Sonata for Solo Violin: IV: Presto,8,Music


**Q36.** Show each customer with their support rep's full name.

In [43]:
pd.read_sql("""
    SELECT Customer.FirstName || ' ' || Customer.LastName AS Name, Employee.EmployeeId ,SupportRepId, Employee.FirstName || ' ' || Employee.LastName AS EmployeeFullName
    FROM Customer
    LEFT JOIN Employee ON Customer.SupportRepId = Employee.EmployeeId

""", conn)

,Name,EmployeeId,SupportRepId,EmployeeFullName
0,Luís Gonçalves,3,3,Jane Peacock
1,Leonie Köhler,5,5,Steve Johnson
2,François Tremblay,3,3,Jane Peacock
3,Bjørn Hansen,4,4,Margaret Park
4,František Wichterlová,4,4,Margaret Park
5,Helena Holý,5,5,Steve Johnson
6,Astrid Gruber,5,5,Steve Johnson
7,Daan Peeters,4,4,Margaret Park
8,Kara Nielsen,4,4,Margaret Park
9,Eduardo Martins,4,4,Margaret Park


**Q37.** Show all invoice lines with the track name, quantity, and unit price.

In [44]:
pd.read_sql("""
    SELECT Track.Name AS TrackName, InvoiceId, Track.UnitPrice as TrackUnitPrice, InvoiceLine.UnitPrice as InvoiceUnitPrice, Quantity
    FROM InvoiceLine
    JOIN Track on InvoiceLine.TrackId = Track.TrackId
""", conn)

,TrackName,InvoiceId,TrackUnitPrice,InvoiceUnitPrice,Quantity
0,Balls to the Wall,1,0.99,0.99,1
1,Restless and Wild,1,0.99,0.99,1
2,Put The Finger On You,2,0.99,0.99,1
3,Inject The Venom,2,0.99,0.99,1
4,Evil Walks,2,0.99,0.99,1
...,...,...,...,...,...
2235,Looking For Love,411,0.99,0.99,1
2236,Sweet Lady Luck,411,0.99,0.99,1
2237,Feirinha da Pavuna/Luz do Repente/Bagaço da La...,411,0.99,0.99,1
2238,Samba pras moças,411,0.99,0.99,1


**Q38.** Show all tracks with their media type name.

In [45]:
pd.read_sql("""
    SELECT Track.Name as TrackName , MediaType.Name AS MediaName
    FROM Track
    JOIN MediaType on Track.MediaTypeId = MediaType.MediaTypeId
""", conn)

,TrackName,MediaName
0,For Those About To Rock (We Salute You),MPEG audio file
1,Balls to the Wall,Protected AAC audio file
2,Fast As a Shark,Protected AAC audio file
3,Restless and Wild,Protected AAC audio file
4,Princess of the Dawn,Protected AAC audio file
...,...,...
3498,Pini Di Roma (Pinien Von Rom) \ I Pini Della V...,Protected AAC audio file
3499,"String Quartet No. 12 in C Minor, D. 703 ""Quar...",Protected AAC audio file
3500,"L'orfeo, Act 3, Sinfonia (Orchestra)",Protected AAC audio file
3501,"Quintet for Horn, Violin, 2 Violas, and Cello ...",Protected AAC audio file


**Q39.** List all playlists with their track count (include playlists with zero tracks).

In [46]:
pd.read_sql("""
    SELECT Playlist.name as playlist, Count(TrackId) AS TrackCount
    FROM Playlist
    LEFT JOIN PlaylistTrack ON Playlist.playlistId = PlaylistTrack.PlaylistId
    GROUP BY Playlist.name
    ORDER BY TrackCount ASC


""", conn)

,playlist,TrackCount
0,Audiobooks,0
1,Movies,0
2,Music Videos,1
3,On-The-Go 1,1
4,Grunge,15
5,Classical 101 - Deep Cuts,25
6,Classical 101 - Next Steps,25
7,Classical 101 - The Basics,25
8,Heavy Metal Classic,26
9,Brazilian Music,39


**Q40.** Show each employee and who they report to (self-join). Include employees who report to nobody.

In [47]:
pd.read_sql("""
    SELECT
        e.FirstName || ' ' || e.LastName AS EmployeeName,
        e.Title,
        m.FirstName || ' ' || m.LastName AS ManagerName
    FROM Employee e
    LEFT JOIN Employee m ON e.ReportsTo = m.EmployeeId
""", conn)

,EmployeeName,Title,ManagerName
0,Andrew Adams,General Manager,None
1,Nancy Edwards,Sales Manager,Andrew Adams
2,Jane Peacock,Sales Support Agent,Nancy Edwards
3,Margaret Park,Sales Support Agent,Nancy Edwards
4,Steve Johnson,Sales Support Agent,Nancy Edwards
5,Michael Mitchell,IT Manager,Andrew Adams
6,Robert King,IT Staff,Michael Mitchell
7,Laura Callahan,IT Staff,Michael Mitchell


**Q41.** Show each customer and the total amount they have spent across all invoices.

In [48]:
pd.read_sql("""
    SELECT c.FirstName || ' ' || c.LastName AS customer, i.Total
    FROM Invoice i
    JOIN CUSTOMER c ON i.CustomerId = c.CustomerId
    ORDER BY i.Total DESC

""", conn)

,customer,Total
0,Helena Holý,25.86
1,Richard Cunningham,23.86
2,Ladislav Kovács,21.86
3,Hugh O'Reilly,21.86
4,Astrid Gruber,18.86
...,...,...
407,Ladislav Kovács,0.99
408,Frank Ralston,0.99
409,François Tremblay,0.99
410,Marc Dubois,0.99


**Q42.** List all tracks in the genre 'Rock', showing track name and album title.

In [49]:
pd.read_sql("""
    SELECT Track.Name AS Track ,Genre.Name AS Genre, Album.Title AS Album
    FROM Track
    JOIN Genre ON Track.GenreId = Genre.GenreId
    JOIN Album ON Track.AlbumId = Album.AlbumId
    WHERE Genre.Name = 'Rock'
""", conn)

,Track,Genre,Album
0,For Those About To Rock (We Salute You),Rock,For Those About To Rock We Salute You
1,Balls to the Wall,Rock,Balls to the Wall
2,Fast As a Shark,Rock,Restless and Wild
3,Restless and Wild,Rock,Restless and Wild
4,Princess of the Dawn,Rock,Restless and Wild
...,...,...,...
1292,Tease Me Please Me,Rock,20th Century Masters - The Millennium Collecti...
1293,Wind of Change,Rock,20th Century Masters - The Millennium Collecti...
1294,Send Me an Angel,Rock,20th Century Masters - The Millennium Collecti...
1295,I Guess You're Right,Rock,Every Kind of Light


**Q43.** Find all artists who have at least one album in the database.

In [50]:
pd.read_sql("""
    SELECT Artist.Name AS Artist, Album.Title AS Album
    FROM Artist
    INNER JOIN Album on Artist.ArtistId = Album.ArtistId
""", conn)

,Artist,Album
0,AC/DC,For Those About To Rock We Salute You
1,Accept,Balls to the Wall
2,Accept,Restless and Wild
3,AC/DC,Let There Be Rock
4,Aerosmith,Big Ones
...,...,...
342,Eugene Ormandy,Respighi:Pines of Rome
343,Emerson String Quartet,Schubert: The Late String Quartets & String Qu...
344,"C. Monteverdi, Nigel Rogers - Chiaroscuro; Lon...",Monteverdi: L'Orfeo
345,Nash Ensemble,Mozart: Chamber Music


**Q44.** Show each employee with the number of customers they support.

In [51]:
pd.read_sql("""
    SELECT Employee.FirstName || ' ' || Employee.LastName AS Employee, Count(*) AS SupportedCustomers
    FROM Employee
    LEFT JOIN Customer on Employee.EmployeeId = Customer.SupportRepId
    Group BY Employee.EmployeeId, Employee.FirstName, Employee.LastName
    ORDER BY SupportedCustomers DESC
""", conn)

,Employee,SupportedCustomers
0,Jane Peacock,21
1,Margaret Park,20
2,Steve Johnson,18
3,Andrew Adams,1
4,Nancy Edwards,1
5,Michael Mitchell,1
6,Robert King,1
7,Laura Callahan,1


**Q45.** List tracks that appear in more than one playlist.

In [52]:
pd.read_sql("""
    SELECT Track.Name AS TrackName, COUNT(PlaylistTrack.PlaylistId) AS PlaylistCount
    FROM Track
    JOIN PlaylistTrack ON Track.TrackId = PlaylistTrack.TrackId
    GROUP BY Track.TrackId, Track.Name
    HAVING COUNT(PlaylistTrack.PlaylistId) > 1;
""", conn)

,TrackName,PlaylistCount
0,For Those About To Rock (We Salute You),3
1,Balls to the Wall,3
2,Fast As a Shark,4
3,Restless and Wild,4
4,Princess of the Dawn,4
...,...,...
3498,Pini Di Roma (Pinien Von Rom) \ I Pini Della V...,5
3499,"String Quartet No. 12 in C Minor, D. 703 ""Quar...",4
3500,"L'orfeo, Act 3, Sinfonia (Orchestra)",4
3501,"Quintet for Horn, Violin, 2 Violas, and Cello ...",4


**Q46.** Find the total revenue generated by each employee through their customers' invoices.

In [53]:
pd.read_sql("""
    SELECT Employee.FirstName || ' ' || Employee.LastName AS Employee, COUNT(Invoice.InvoiceId) AS TotalInvoices, SUM(Invoice.Total) AS TotalRevenue
    FROM Employee
    JOIN Customer ON Employee.EmployeeId = Customer.SupportRepId
    JOIN Invoice ON Customer.CustomerId = Invoice.CustomerId
    GROUP BY Employee.EmployeeId, Employee.FirstName, Employee.LastName

""", conn)

,Employee,TotalInvoices,TotalRevenue
0,Jane Peacock,146,833.04
1,Margaret Park,140,775.40
2,Steve Johnson,126,720.16


**Q47.** Find the most popular genre by total number of tracks sold (from InvoiceLine).

In [54]:
pd.read_sql("""
    SELECT Genre.Name AS Genre , Count (InvoiceLine.InvoiceLineId) AS TrackSoldCount
    FROM Genre
    JOIN Track ON Genre.GenreId = Track.GenreId
    JOIN InvoiceLine ON Track.TrackId = InvoiceLine.TrackId
    GROUP BY Genre.Name
    ORDER BY TrackSoldCount DESC

""", conn)

,Genre,TrackSoldCount
0,Rock,835
1,Latin,386
2,Metal,264
3,Alternative & Punk,244
4,Jazz,80
5,Blues,61
6,TV Shows,47
7,R&B/Soul,41
8,Classical,41
9,Reggae,30


**Q48.** List all tracks that have NEVER appeared on an invoice (never purchased).

In [55]:
pd.read_sql("""
    SELECT TRACK.Name AS Track, InvoiceLine.InvoiceId
    FROM TRACK
    LEFT JOIN InvoiceLine ON TRACK.TrackId = InvoiceLine.TrackId
    WHERE InvoiceLine.InvoiceId IS NULL
""", conn)

,Track,InvoiceId
0,Let's Get It Up,None
1,C.O.D.,None
2,Let There Be Rock,None
3,Bad Boy Boogie,None
4,Whole Lotta Rosie,None
...,...,...
1514,"Erlkonig, D.328",None
1515,"Concerto for Violin, Strings and Continuo in G...",None
1516,"L'orfeo, Act 3, Sinfonia (Orchestra)",None
1517,"Quintet for Horn, Violin, 2 Violas, and Cello ...",None


**Q49.** Show the top 5 best-selling tracks by total quantity sold.

In [56]:
pd.read_sql("""
    SELECT TRACK.Name AS Track, SUM(InvoiceLine.Quantity) AS InvoiceCount
    FROM TRACK
    JOIN InvoiceLine ON TRACK.TrackId = InvoiceLine.TrackId
    GROUP BY TRACK.TrackId, TRACK.Name
    ORDER BY InvoiceCount DESC
    LIMIT 5

""", conn)

,Track,InvoiceCount
0,Balls to the Wall,2
1,Inject The Venom,2
2,Snowballed,2
3,Overdose,2
4,Deuces Are Wild,2


**Q50.** List artists whose tracks have never been purchased.

In [57]:
pd.read_sql("""
    SELECT ar.Name AS ArtistName, il.InvoiceId
    FROM Artist ar
    LEFT JOIN Album al ON ar.ArtistId = al.ArtistId
    LEFT JOIN Track t ON al.AlbumId = t.AlbumId
    LEFT JOIN InvoiceLine il ON t.TrackId = il.TrackId
    WHERE il.InvoiceLineId IS NULL;
""", conn)

,ArtistName,InvoiceId
0,AC/DC,None
1,AC/DC,None
2,AC/DC,None
3,AC/DC,None
4,AC/DC,None
...,...,...
1585,Gerald Moore,None
1586,"Mela Tenenbaum, Pro Musica Prague & Richard Kapp",None
1587,"C. Monteverdi, Nigel Rogers - Chiaroscuro; Lon...",None
1588,Nash Ensemble,None


---
## Level 4 — Subqueries (Q51–65)
Scalar subqueries, IN / NOT IN, EXISTS, correlated subqueries.

**Q51.** Find the customer who has spent the most overall.

In [58]:
pd.read_sql("""
    SELECT CUSTOMER.FirstName || ' ' || CUSTOMER.LastName AS Customer, SUM(Invoice.Total) AS Total
    FROM Invoice
    JOIN CUSTOMER ON Invoice.CustomerId = CUSTOMER.CustomerId
    GROUP BY Customer, Invoice.InvoiceId
    ORDER BY Total DESC

""", conn)

,Customer,Total
0,Helena Holý,25.86
1,Richard Cunningham,23.86
2,Ladislav Kovács,21.86
3,Hugh O'Reilly,21.86
4,Astrid Gruber,18.86
...,...,...
407,Ladislav Kovács,0.99
408,Frank Ralston,0.99
409,François Tremblay,0.99
410,Marc Dubois,0.99


**Q52.** Find all customers who have spent more than the average customer spending.

In [59]:
pd.read_sql("""
    SELECT AVG(Total) FROM Invoice
""", conn)


,AVG(Total)
0,5.651942


In [60]:

pd.read_sql("""
    SELECT CUSTOMER.FirstName || ' ' || CUSTOMER.LastName AS Customer, Invoice.InvoiceId, Invoice.Total
    FROM Invoice
    JOIN CUSTOMER ON Invoice.CustomerId = CUSTOMER.CustomerId
    WHERE Invoice.Total > (SELECT AVG(Total) FROM Invoice)
    ORDER BY Invoice.Total ASC

""", conn)

,Customer,InvoiceId,Total
0,Daan Peeters,3,5.94
1,Hugh O'Reilly,10,5.94
2,Victor Stevens,17,5.94
3,Bjørn Hansen,24,5.94
4,Wyatt Girard,31,5.94
...,...,...,...
174,Victor Stevens,201,18.86
175,Ladislav Kovács,96,21.86
176,Hugh O'Reilly,194,21.86
177,Richard Cunningham,299,23.86


**Q53.** Find the artist with the most tracks in the database.

In [61]:
pd.read_sql("""
    SELECT Artist.Name AS Artist, Track.Name AS Track, SUM(Track.Name) AS TrackCount
    FROM TRACK
    JOIN Album ON Track.AlbumId = Album.AlbumId
    JOIN Artist ON Album.ArtistId = Artist.ArtistId
    Group BY Artist.Name, Track.Name
    Order BY TrackCount DESC
    Limit 1
""", conn)

,Artist,Track,TrackCount
0,Rush,2112 Overture,2112.0


**Q54.** Which country has the most customers? Use a subquery.

In [62]:
pd.read_sql("""
    SELECT Country, COUNT(CustomerId) AS CustomerCount
    FROM Customer
    GROUP BY Country

    HAVING COUNT(CustomerId) = (
    SELECT MAX(CountryCount)
    FROM (
        SELECT COUNT(CustomerId) AS CountryCount
        FROM Customer
        GROUP BY Country
    ) AS SubTable
    )
    ;

""", conn)

,Country,CustomerCount
0,USA,13


**Q55.** Find the longest track in each genre using a subquery.

In [63]:
pd.read_sql("""
    SELECT TRACK.Name As Track , MAX(Milliseconds) AS MaxMilliseconds
    FROM TRACK

""", conn)

,Track,MaxMilliseconds
0,Occupation / Precipice,5286953


**Q56.** Find customers who have never bought a Rock track. (Hint: use NOT IN or NOT EXISTS)

In [64]:
pd.read_sql("""
    SELECT CustomerId, FirstName, LastName
    FROM Customer
    WHERE CustomerId NOT IN (
        SELECT DISTINCT i.CustomerId
        FROM Invoice i
        JOIN InvoiceLine il ON i.InvoiceId = il.InvoiceId
        JOIN Track t ON il.TrackId = t.TrackId
        JOIN Genre g ON t.GenreId = g.GenreId
        WHERE g.Name = 'Rock'
    );

""", conn)

,CustomerId,FirstName,LastName


**Q57.** Find the top 3 selling genres by total revenue.

In [65]:
pd.read_sql("""
    SELECT g.Name AS GenreName,
    SUM(il.UnitPrice * il.Quantity) AS TotalRevenue
    FROM Genre g
    JOIN Track t ON g.GenreId = t.GenreId
    JOIN InvoiceLine il ON t.TrackId = il.TrackId
    GROUP BY g.GenreId
    ORDER BY TotalRevenue DESC
    LIMIT 3;
""", conn)

,GenreName,TotalRevenue
0,Rock,826.65
1,Latin,382.14
2,Metal,261.36


**Q58.** List tracks that are in more than 3 playlists.

In [66]:
pd.read_sql("""
    SELECT Track.Name AS Track, Count(PlaylistTrack.PlaylistId) AS PlaylistCount
    FROM Track
    JOIN PlaylistTrack ON Track.TrackId = PlaylistTrack.TrackId
    GROUP BY Track.TrackId, Track.Name
    HAVING Count(PlaylistTrack.PlaylistId) > 3

""", conn)

,Track,PlaylistCount
0,Fast As a Shark,4
1,Restless and Wild,4
2,Princess of the Dawn,4
3,Man In The Box,4
4,Sozinho,4
...,...,...
106,Pini Di Roma (Pinien Von Rom) \ I Pini Della V...,5
107,"String Quartet No. 12 in C Minor, D. 703 ""Quar...",4
108,"L'orfeo, Act 3, Sinfonia (Orchestra)",4
109,"Quintet for Horn, Violin, 2 Violas, and Cello ...",4


**Q59.** Find the most popular artist by total revenue from track sales.

In [67]:
pd.read_sql("""
    SELECT  Artist.Name AS Artist, SUM(Invoice.Total) AS Total
    FROM Artist
    JOIN Album ON Artist.ArtistId = Album.ArtistId
    JOIN Track ON Album.AlbumId = Track.AlbumId
    JOIN InvoiceLine ON Track.TrackId = InvoiceLine.TrackId
    JOIN Invoice ON InvoiceLine.InvoiceId = Invoice.InvoiceId
    GROUP BY Artist.Name
    ORDER BY Total DESC

""", conn)

,Artist,Total
0,Iron Maiden,1233.54
1,U2,895.59
2,Lost,833.70
3,Led Zeppelin,620.73
4,Metallica,599.94
...,...,...
160,Orchestra of The Age of Enlightenment,1.98
161,"Emanuel Ax, Eugene Ormandy & Philadelphia Orch...",1.98
162,Berliner Philharmoniker & Hans Rosbaud,1.98
163,Academy of St. Martin in the Fields & Sir Nevi...,1.98


**Q60.** Find the month (across all years) with the highest total invoice revenue.

In [68]:
pd.read_sql("""
SELECT
    STRFTIME('%m', InvoiceDate) AS Month, -- extracts month number
    SUM(Total) AS TotalRevenue
FROM Invoice
GROUP BY Month
ORDER BY TotalRevenue DESC


""", conn)

,Month,TotalRevenue
0,01,201.12
1,06,201.10
2,04,198.14
3,08,198.10
4,09,196.20
5,03,195.10
6,10,193.10
7,05,193.10
8,07,190.10
9,12,189.10


**Q61.** What percentage of total revenue does each country contribute? Round to 2 decimal places.

In [69]:
pd.read_sql("""
 SELECT
    BillingCountry,
    ROUND(SUM(Total) * 100.0 / (SELECT SUM(Total) FROM Invoice), 2) AS PercentageContribution
FROM
    Invoice
GROUP BY
    BillingCountry
ORDER BY
    PercentageContribution DESC;
""", conn)

,BillingCountry,PercentageContribution
0,USA,22.46
1,Canada,13.05
2,France,8.38
3,Brazil,8.16
4,Germany,6.72
5,United Kingdom,4.85
6,Czech Republic,3.88
7,Portugal,3.32
8,India,3.23
9,Chile,2.00


**Q62.** Find employees who manage at least one other employee.

In [70]:
pd.read_sql("""
    SELECT EmployeeId, FirstName, LastName, Title
    FROM Employee
    WHERE EmployeeId IN (
        SELECT DISTINCT ReportsTo
        FROM Employee
        WHERE ReportsTo IS NOT NULL
    );
""", conn)

,EmployeeId,FirstName,LastName,Title
0,1,Andrew,Adams,General Manager
1,2,Nancy,Edwards,Sales Manager
2,6,Michael,Mitchell,IT Manager


**Q63.** Find the artist whose tracks appear in the most playlists (count distinct playlists).

In [71]:
pd.read_sql("""
   SELECT
    a.Name AS ArtistName,
    COUNT(DISTINCT pt.PlaylistId) AS PlaylistCount
    FROM Artist a
    JOIN Album al ON a.ArtistId = al.ArtistId
    JOIN Track t ON al.AlbumId = t.AlbumId
    JOIN PlaylistTrack pt ON t.TrackId = pt.TrackId
    GROUP BY a.ArtistId, a.Name
    ORDER BY PlaylistCount DESC
    LIMIT 1;
""", conn)

,ArtistName,PlaylistCount
0,Eugene Ormandy,7


**Q64.** Find the most recent invoice date for each customer.

In [72]:
pd.read_sql("""
    SELECT CUSTOMER.FirstName || ' ' || CUSTOMER.LastName AS Customer,
    InvoiceId, MAX(InvoiceDate) AS MostRecentInvoiceDate
    FROM Invoice
    JOIN CUSTOMER ON Invoice.CustomerId = CUSTOMER.CustomerId
    Group BY Customer
    ORDER BY InvoiceDate DESC


""", conn)

,Customer,InvoiceId,MostRecentInvoiceDate
0,Manoj Pareek,412,2025-12-22 00:00:00
1,Terhi Hämäläinen,411,2025-12-14 00:00:00
2,Madalena Sampaio,410,2025-12-09 00:00:00
3,Robert Brown,409,2025-12-06 00:00:00
4,Victor Stevens,408,2025-12-05 00:00:00
5,Kathy Chase,406,2025-12-04 00:00:00
6,John Gordon,407,2025-12-04 00:00:00
7,Dan Miller,405,2025-11-21 00:00:00
8,Helena Holý,404,2025-11-13 00:00:00
9,Diego Gutiérrez,403,2025-11-08 00:00:00


**Q65.** Find the track that generated the most total revenue (UnitPrice × Quantity across all invoice lines).

In [73]:
pd.read_sql("""
    SELECT t.Name AS TrackName, SUM(il.UnitPrice * il.Quantity) AS TotalRevenue
    FROM Track t
    JOIN InvoiceLine il ON t.TrackId = il.TrackId
    GROUP BY t.TrackId, t.Name
    ORDER BY TotalRevenue DESC
    LIMIT 1;

""", conn)

,TrackName,TotalRevenue
0,The Woman King,3.98


---
## Level 5 — CTEs (Q66–80)
WITH ... AS (...), chained CTEs, readable multi-step logic.

**Q66.** Using a CTE, find all artists with more albums than the average number of albums per artist.

In [74]:
pd.read_sql("""
  WITH ArtistAlbumCounts AS (
      SELECT
          ArtistId,
          COUNT(AlbumId) AS AlbumCount
      FROM Album
      GROUP BY ArtistId
  )
  SELECT ArtistId, AlbumCount
  FROM ArtistAlbumCounts
  WHERE AlbumCount > (SELECT AVG(AlbumCount) FROM ArtistAlbumCounts)
""", conn)


,ArtistId,AlbumCount
0,1,2
1,2,2
2,6,2
3,8,3
4,11,2
5,12,2
6,16,2
7,18,2
8,19,2
9,21,4


**Q67.** Using a CTE, calculate each customer's total spending and their share (%) of their country's total spending.

In [75]:
pd.read_sql("""
   WITH CustomerSpending AS (
    -- CTE 1: Calculate total spending for each customer
    SELECT
        CustomerId,
        BillingCountry AS Country,
        SUM(Total) AS TotalCustomerSpent
    FROM Invoice
    GROUP BY CustomerId, BillingCountry
),
CountrySpending AS (
    -- CTE 2: Calculate total spending for each country
    SELECT
        Country,
        SUM(TotalCustomerSpent) AS TotalCountrySpent
    FROM CustomerSpending
    GROUP BY Country
)
-- Final Select: Join them and calculate the percentage share
SELECT
    cs.CustomerId,
    cs.Country,
    cs.TotalCustomerSpent,
    ROUND((cs.TotalCustomerSpent / cns.TotalCountrySpent) * 100, 2) AS SharePercentage
FROM CustomerSpending cs
JOIN CountrySpending cns ON cs.Country = cns.Country
ORDER BY cs.Country, SharePercentage DESC;
""", conn)



,CustomerId,Country,TotalCustomerSpent,SharePercentage
0,56,Argentina,37.62,100.00
1,55,Australia,37.62,100.00
2,7,Austria,42.62,100.00
3,8,Belgium,37.62,100.00
4,1,Brazil,39.62,20.84
5,10,Brazil,37.62,19.79
6,11,Brazil,37.62,19.79
7,12,Brazil,37.62,19.79
8,13,Brazil,37.62,19.79
9,3,Canada,39.62,13.03


**Q68.** Using a CTE, find the top-selling track within each genre (by quantity sold).

In [76]:
pd.read_sql("""
    WITH TrackSales AS (
        SELECT
            g.Name AS GenreName,
            t.Name AS TrackName,
            SUM(il.Quantity) AS TotalSold
        FROM InvoiceLine il
        JOIN Track t ON il.TrackId = t.TrackId
        JOIN Genre g ON t.GenreId = g.GenreId
        GROUP BY g.Name, t.Name
    ),
    RankedSales AS (
        SELECT
            GenreName,
            TrackName,
            TotalSold,
            RANK() OVER (PARTITION BY GenreName ORDER BY TotalSold DESC) as SalesRank
        FROM TrackSales
    )
    SELECT
        GenreName,
        TrackName,
        TotalSold
    FROM RankedSales
    WHERE SalesRank = 1
    ORDER BY GenreName;

""", conn)

,GenreName,TrackName,TotalSold
0,Alternative,All Night Thing,1
1,Alternative,Billie Jean,1
2,Alternative,Call Me a Dog,1
3,Alternative,Disappearing Act,1
4,Alternative,Four Walled World,1
...,...,...,...
134,World,No Futuro,1
135,World,O Que Vai Em Meu Coração,1
136,World,Papelão,1
137,World,Voce Inteira,1


**Q69.** Using a CTE, find employees whose customers generate above-average total revenue.

In [77]:
pd.read_sql("""

   WITH EmployeeRevenue AS (
    -- Step 1: Calculate total revenue per employee
    SELECT
        e.EmployeeId,
        e.FirstName || ' ' || e.LastName AS EmployeeName,
        SUM(i.Total) AS EmployeeTotalRevenue
    FROM Employee e
    JOIN Customer c ON e.EmployeeId = c.SupportRepId
    JOIN Invoice i ON c.CustomerId = i.CustomerId
    GROUP BY e.EmployeeId, EmployeeName
)
SELECT *
FROM EmployeeRevenue
WHERE EmployeeTotalRevenue > (SELECT AVG(EmployeeTotalRevenue) FROM EmployeeRevenue)
ORDER BY EmployeeTotalRevenue DESC;

""", conn)

,EmployeeId,EmployeeName,EmployeeTotalRevenue
0,3,Jane Peacock,833.04


**Q70.** Using a CTE, find 'loyal' customers who made purchases in 3 or more distinct years.

In [78]:
pd.read_sql("""
    WITH CustomerLoyalty AS (
        SELECT
            CustomerId,
            COUNT(DISTINCT strftime('%Y', InvoiceDate)) AS YearsActive
        FROM Invoice
        GROUP BY CustomerId
    )
    SELECT
        CustomerLoyalty.CustomerId,
        YearsActive,
        Customer.FirstName || ' ' || Customer.LastName AS CustomerName
    FROM CustomerLoyalty
    JOIN Customer ON CustomerLoyalty.CustomerId = Customer.CustomerId
    WHERE YearsActive >= 3
    ORDER BY YearsActive DESC;
""", conn)

,CustomerId,YearsActive,CustomerName
0,5,5,František Wichterlová
1,10,5,Eduardo Martins
2,16,5,Frank Harris
3,21,5,Kathy Chase
4,26,5,Richard Cunningham
5,32,5,Aaron Mitchell
6,37,5,Fynn Zimmermann
7,48,5,Johannes Van der Berg
8,53,5,Phil Hughes
9,1,4,Luís Gonçalves


**Q71.** Using a CTE, compute the total revenue per year and show the year-over-year change in revenue.

In [79]:
pd.read_sql("""
WITH YearlyRevenue AS (
    SELECT
        strftime('%Y', InvoiceDate) AS SalesYear,
        -- Create a "MatchYear" by adding 1 to the current year
        strftime('%Y', InvoiceDate, '+1 year') AS MatchYear,
        SUM(Total) AS AnnualRevenue
    FROM Invoice
    GROUP BY SalesYear
)
SELECT
    curr.SalesYear,
    curr.AnnualRevenue AS CurrentRevenue,
    prev.AnnualRevenue AS PreviousRevenue,
    ROUND(curr.AnnualRevenue - prev.AnnualRevenue, 2) AS YoY_Difference
FROM YearlyRevenue curr
LEFT JOIN YearlyRevenue prev
    ON curr.SalesYear = prev.MatchYear
ORDER BY curr.SalesYear;

""", conn)

,SalesYear,CurrentRevenue,PreviousRevenue,YoY_Difference
0,2021,449.46,NaN,NaN
1,2022,481.45,449.46,31.99
2,2023,469.58,481.45,-11.87
3,2024,477.53,469.58,7.95
4,2025,450.58,477.53,-26.95


**Q72.** Using a CTE, find albums where every single track has been purchased at least once.

In [80]:
pd.read_sql("""
 WITH AlbumSales AS (
    SELECT
        t.AlbumId,
        COUNT(t.TrackId) AS TotalTracks,
        COUNT(DISTINCT il.TrackId) AS SoldTracks
    FROM Track t
    LEFT JOIN InvoiceLine il ON t.TrackId = il.TrackId
    GROUP BY t.AlbumId
)
SELECT a.Title
FROM Album a
JOIN AlbumSales AS s ON a.AlbumId = s.AlbumId
WHERE s.TotalTracks = s.SoldTracks;

""", conn)

,Title
0,Restless and Wild
1,BBC Sessions [Disc 2] [Live]
2,Bark at the Moon (Remastered)
3,Un-Led-Ed
4,Duos II
5,Pachelbel: Canon & Gigue
6,Bach: The Cello Suites
7,Handel: The Messiah (Highlights)
8,Haydn: Symphonies 99 - 104
9,Wagner: Favourite Overtures


**Q73.** Using two CTEs, find the second most popular genre by total tracks sold.

In [81]:
pd.read_sql("""
    WITH GenreSales AS (
    -- CTE 1: Calculate total sales per genre
    SELECT
        g.Name AS GenreName,
        SUM(il.Quantity) AS TotalSold
    FROM InvoiceLine il
    JOIN Track t ON il.TrackId = t.TrackId
    JOIN Genre g ON t.GenreId = g.GenreId
    GROUP BY g.Name
),
OrderedSales AS (
    -- CTE 2: Simply order the results
    SELECT GenreName, TotalSold
    FROM GenreSales
    ORDER BY TotalSold DESC
)
-- Final Select: Skip the 1st (top) row and take the next 1
SELECT GenreName, TotalSold
FROM OrderedSales
LIMIT 1 OFFSET 1;

""", conn)

,GenreName,TotalSold
0,Latin,386


**Q74.** Using a CTE, group customers by the year of their first purchase and show total spend per cohort.

In [82]:
pd.read_sql("""
 -- Find the first purchase year for each customer

WITH CustomerCohorts AS (
    SELECT
        CustomerId,
        MIN(strftime('%Y', InvoiceDate)) AS FirstYear
    FROM Invoice
    GROUP BY CustomerId
)
-- Join cohorts with all invoices to sum spending
SELECT
    cc.FirstYear AS Cohort,
    ROUND(SUM(i.Total), 2) AS TotalSpend,
    COUNT(DISTINCT cc.CustomerId) AS CustomerCount
FROM CustomerCohorts cc
JOIN Invoice i ON cc.CustomerId = i.CustomerId
GROUP BY cc.FirstYear
ORDER BY cc.FirstYear;

""", conn)

,Cohort,TotalSpend,CustomerCount
0,2021,1812.54,46
1,2022,516.06,13


**Q75.** Using a CTE, find the average number of tracks per album for each artist.

In [83]:
pd.read_sql("""
    WITH AlbumTrackCounts AS (
        SELECT
            alb.ArtistId,
            t.AlbumId,
            COUNT(t.TrackId) AS TrackCount
        FROM Track t
        JOIN Album alb ON t.AlbumId = alb.AlbumId -- Link Track to Artist via Album
        GROUP BY alb.ArtistId, t.AlbumId
    )
    SELECT
        art.Name AS ArtistName,
        AVG(atc.TrackCount) AS AvgTracksPerAlbum
    FROM Artist art
    JOIN AlbumTrackCounts atc ON art.ArtistId = atc.ArtistId
    GROUP BY art.ArtistId, art.Name
    ORDER BY AvgTracksPerAlbum DESC;
""", conn)


,ArtistName,AvgTracksPerAlbum
0,Lenny Kravitz,57.0
1,Chico Buarque,34.0
2,Eric Clapton,24.0
3,Frank Sinatra,24.0
4,Battlestar Galactica (Classic),24.0
...,...,...
199,"Mela Tenenbaum, Pro Musica Prague & Richard Kapp",1.0
200,Emerson String Quartet,1.0
201,"C. Monteverdi, Nigel Rogers - Chiaroscuro; Lon...",1.0
202,Nash Ensemble,1.0


**Q76.** Using a CTE, find all pairs of customers from the same city.

In [84]:
pd.read_sql("""
    WITH CustomerList AS (
    SELECT CustomerId, FirstName || ' ' || LastName AS FullName, City
    FROM Customer
)
SELECT
    c1.City,
    c1.FullName AS Customer1,
    c2.FullName AS Customer2
FROM CustomerList c1
JOIN CustomerList c2 ON c1.City = c2.City
WHERE c1.CustomerId < c2.CustomerId
ORDER BY c1.City;


""", conn)

,City,Customer1,Customer2
0,Berlin,Hannah Schneider,Niklas Schröder
1,London,Emma Jones,Phil Hughes
2,Mountain View,Frank Harris,Dan Miller
3,Paris,Camille Bernard,Dominique Lefebvre
4,Prague,František Wichterlová,Helena Holý
5,São Paulo,Eduardo Martins,Alexandre Rocha


**Q77.** Using a CTE, find tracks that were purchased by more than one distinct customer.

In [85]:
pd.read_sql("""
    WITH TrackCustomerCounts AS (
    -- Step 1: Count unique customers for each track
    SELECT
        TrackId,
        COUNT(DISTINCT CustomerId) AS UniqueCustomerCount
    FROM InvoiceLine il
    JOIN Invoice i ON il.InvoiceId = i.InvoiceId
    GROUP BY TrackId
)
-- Step 2: Join with Track table to get names and filter
SELECT
    t.Name AS TrackName,
    tcc.UniqueCustomerCount
FROM Track t
JOIN TrackCustomerCounts tcc ON t.TrackId = tcc.TrackId
WHERE tcc.UniqueCustomerCount > 1
ORDER BY tcc.UniqueCustomerCount DESC;

""", conn)

,TrackName,UniqueCustomerCount
0,Balls to the Wall,2
1,Inject The Venom,2
2,Snowballed,2
3,Overdose,2
4,Deuces Are Wild,2
...,...,...
251,"Symphonie Fantastique, Op. 14: V. Songe d'une ...",2
252,Rehab,2
253,"Suite No. 3 in D, BWV 1068: III. Gavotte I & II",2
254,"Music for the Funeral of Queen Mary: VI. ""Thou...",2


**Q78.** Using a CTE, find the top 3 most purchased tracks per billing country.

In [86]:
pd.read_sql("""
    WITH TrackSalesByCountry AS (
    -- Step 1: Calculate total sales for each track per country
    SELECT
        i.BillingCountry,
        t.Name AS TrackName,
        SUM(il.Quantity) AS TotalPurchased
    FROM InvoiceLine il
    JOIN Invoice i ON il.InvoiceId = i.InvoiceId
    JOIN Track t ON il.TrackId = t.TrackId
    GROUP BY i.BillingCountry, t.Name
),
RankedTracks AS (
    -- Step 2: Rank tracks within each country
    SELECT
        BillingCountry,
        TrackName,
        TotalPurchased,
        ROW_NUMBER() OVER (PARTITION BY BillingCountry ORDER BY TotalPurchased DESC, TrackName ASC) AS SalesRank
    FROM TrackSalesByCountry
)
-- Step 3: Filter for top 3
SELECT
    BillingCountry,
    TrackName,
    TotalPurchased
FROM RankedTracks
WHERE SalesRank <= 3
ORDER BY BillingCountry, SalesRank;


""", conn)

,BillingCountry,TrackName,TotalPurchased
0,Argentina,Battery,1
1,Argentina,Better Than You,1
2,Argentina,Cabeça Dinossauro,1
3,Australia,09 - Iron Maiden,1
4,Australia,2 A.M.,1
...,...,...,...
67,USA,Branch Closing,2
68,USA,Gay Witch Hunt,2
69,United Kingdom,Firmamento,2
70,United Kingdom,2 X 4,1


**Q79.** Using a recursive CTE, display each employee's full management chain up to the CEO.

In [87]:
pd.read_sql("""
    WITH RECURSIVE ManagementChain AS (
    -- Anchor: Start with every employee
    SELECT
        EmployeeId,
        FirstName || ' ' || LastName AS EmployeeName,
        ReportsTo,
        FirstName || ' ' || LastName AS Path
    FROM Employee

    UNION ALL

    -- Recursive Step: Join current results to their managers
    SELECT
        mc.EmployeeId,
        mc.EmployeeName,
        e.ReportsTo,
        mc.Path || ' -> ' || e.FirstName || ' ' || e.LastName
    FROM ManagementChain mc
    JOIN Employee e ON mc.ReportsTo = e.EmployeeId
)
-- Final Select: Filter for rows that have reached the CEO (ReportsTo is NULL)
SELECT
    EmployeeName,
    Path AS ManagementPath
FROM ManagementChain
WHERE ReportsTo IS NULL
ORDER BY EmployeeName;

""", conn)

,EmployeeName,ManagementPath
0,Andrew Adams,Andrew Adams
1,Jane Peacock,Jane Peacock -> Nancy Edwards -> Andrew Adams
2,Laura Callahan,Laura Callahan -> Michael Mitchell -> Andrew A...
3,Margaret Park,Margaret Park -> Nancy Edwards -> Andrew Adams
4,Michael Mitchell,Michael Mitchell -> Andrew Adams
5,Nancy Edwards,Nancy Edwards -> Andrew Adams
6,Robert King,Robert King -> Michael Mitchell -> Andrew Adams
7,Steve Johnson,Steve Johnson -> Nancy Edwards -> Andrew Adams


**Q80.** Using a CTE, compute a 'diversity score' per customer = number of distinct genres they've purchased.

In [88]:
pd.read_sql("""
    WITH CustomerGenres AS (
    -- Step 1: Link each customer to the genres they have purchased
    SELECT DISTINCT
        i.CustomerId,
        t.GenreId
    FROM Invoice i
    JOIN InvoiceLine il ON i.InvoiceId = il.InvoiceId
    JOIN Track t ON il.TrackId = t.TrackId
)
-- Step 2: Count the distinct genres per customer
SELECT
    c.FirstName || ' ' || c.LastName AS CustomerName,
    COUNT(cg.GenreId) AS DiversityScore
FROM Customer c
JOIN CustomerGenres cg ON c.CustomerId = cg.CustomerId
GROUP BY c.CustomerId, CustomerName
ORDER BY DiversityScore DESC;


""", conn)

,CustomerName,DiversityScore
0,Luis Rojas,12
1,Ladislav Kovács,11
2,François Tremblay,10
3,Mark Philips,10
4,Jack Smith,10
5,Frank Ralston,10
6,João Fernandes,10
7,Fynn Zimmermann,10
8,Helena Holý,9
9,Astrid Gruber,9


---
## Level 6 — Window Functions (Q81–100)
ROW_NUMBER, RANK, DENSE_RANK, LAG, LEAD, running totals, moving averages.

**Q81.** Rank customers by total spending using RANK(). Show rank, name, and total spent.

In [89]:
pd.read_sql("""
    SELECT
    RANK() OVER (ORDER BY SUM(i.Total) DESC) AS SpendingRank,
    c.FirstName || ' ' || c.LastName AS CustomerName,
    ROUND(SUM(i.Total), 2) AS TotalSpent
    FROM Customer c
    JOIN Invoice i ON c.CustomerId = i.CustomerId
    GROUP BY c.CustomerId, CustomerName
    ORDER BY SpendingRank;


""", conn)

,SpendingRank,CustomerName,TotalSpent
0,1,Helena Holý,49.62
1,2,Richard Cunningham,47.62
2,3,Luis Rojas,46.62
3,4,Ladislav Kovács,45.62
4,4,Hugh O'Reilly,45.62
5,6,Julia Barnett,43.62
6,7,Frank Ralston,43.62
7,7,Fynn Zimmermann,43.62
8,9,Astrid Gruber,42.62
9,9,Victor Stevens,42.62


**Q82.** Compute a running total of invoice revenue ordered by InvoiceDate.

In [90]:
pd.read_sql("""
    SELECT
    InvoiceId,
    InvoiceDate,
    Total AS InvoiceAmount,
    SUM(Total) OVER (ORDER BY InvoiceDate, InvoiceId) AS RunningTotal
    FROM Invoice
    ORDER BY InvoiceDate, InvoiceId;
""", conn)

,InvoiceId,InvoiceDate,InvoiceAmount,RunningTotal
0,1,2021-01-01 00:00:00,1.98,1.98
1,2,2021-01-02 00:00:00,3.96,5.94
2,3,2021-01-03 00:00:00,5.94,11.88
3,4,2021-01-06 00:00:00,8.91,20.79
4,5,2021-01-11 00:00:00,13.86,34.65
...,...,...,...,...
407,408,2025-12-05 00:00:00,3.96,2297.90
408,409,2025-12-06 00:00:00,5.94,2303.84
409,410,2025-12-09 00:00:00,8.91,2312.75
410,411,2025-12-14 00:00:00,13.86,2326.61


**Q83.** Rank tracks within each album by length (longest = rank 1) using ROW_NUMBER().

In [91]:
pd.read_sql("""
    SELECT
    AlbumId,
    Name AS TrackName,
    Milliseconds,
    ROW_NUMBER() OVER (
        PARTITION BY AlbumId
        ORDER BY Milliseconds DESC
    ) AS TrackRank
FROM Track
ORDER BY AlbumId, TrackRank;


""", conn)

,AlbumId,TrackName,Milliseconds,TrackRank
0,1,For Those About To Rock (We Salute You),343719,1
1,1,Spellbound,270863,2
2,1,Evil Walks,263497,3
3,1,Breaking The Rules,263288,4
4,1,Let's Get It Up,233926,5
...,...,...,...,...
3498,343,Pini Di Roma (Pinien Von Rom) \ I Pini Della V...,286741,1
3499,344,"String Quartet No. 12 in C Minor, D. 703 ""Quar...",139200,1
3500,345,"L'orfeo, Act 3, Sinfonia (Orchestra)",66639,1
3501,346,"Quintet for Horn, Violin, 2 Violas, and Cello ...",221331,1


**Q84.** For each invoice, show the previous invoice total for the same customer using LAG().

In [92]:
pd.read_sql("""
    SELECT
    CustomerId,
    InvoiceId,
    InvoiceDate,
    Total AS CurrentInvoiceTotal,
    LAG(Total) OVER (
        PARTITION BY CustomerId
        ORDER BY InvoiceDate, InvoiceId
    ) AS PreviousInvoiceTotal
FROM Invoice
ORDER BY CustomerId, InvoiceDate;



""", conn)

,CustomerId,InvoiceId,InvoiceDate,CurrentInvoiceTotal,PreviousInvoiceTotal
0,1,98,2022-03-11 00:00:00,3.98,NaN
1,1,121,2022-06-13 00:00:00,3.96,3.98
2,1,143,2022-09-15 00:00:00,5.94,3.96
3,1,195,2023-05-06 00:00:00,0.99,5.94
4,1,316,2024-10-27 00:00:00,1.98,0.99
...,...,...,...,...,...
407,59,45,2021-07-08 00:00:00,5.94,3.96
408,59,97,2022-02-26 00:00:00,1.99,5.94
409,59,218,2023-08-20 00:00:00,1.98,1.99
410,59,229,2023-09-30 00:00:00,13.86,1.98


**Q85.** For each invoice, show the next invoice total for the same customer using LEAD().

In [93]:
pd.read_sql("""
    SELECT
    CustomerId,
    InvoiceId,
    InvoiceDate,
    Total AS CurrentInvoiceTotal,
    LEAD(Total) OVER (
        PARTITION BY CustomerId
        ORDER BY InvoiceDate, InvoiceId
    ) AS NextInvoiceTotal
FROM Invoice
ORDER BY CustomerId, InvoiceDate;


""", conn)

,CustomerId,InvoiceId,InvoiceDate,CurrentInvoiceTotal,NextInvoiceTotal
0,1,98,2022-03-11 00:00:00,3.98,3.96
1,1,121,2022-06-13 00:00:00,3.96,5.94
2,1,143,2022-09-15 00:00:00,5.94,0.99
3,1,195,2023-05-06 00:00:00,0.99,1.98
4,1,316,2024-10-27 00:00:00,1.98,13.86
...,...,...,...,...,...
407,59,45,2021-07-08 00:00:00,5.94,1.99
408,59,97,2022-02-26 00:00:00,1.99,1.98
409,59,218,2023-08-20 00:00:00,1.98,13.86
410,59,229,2023-09-30 00:00:00,13.86,8.91


**Q86.** Show the top-ranked customer (by spend) in each country using RANK() OVER (PARTITION BY Country).

In [94]:
pd.read_sql("""
    WITH CustomerSpending AS (
    SELECT
        BillingCountry AS Country,
        FirstName || ' ' || LastName AS CustomerName,
        SUM(Total) AS TotalSpent
    FROM Invoice i
    JOIN Customer c ON i.CustomerId = c.CustomerId
    GROUP BY c.CustomerId, Country, CustomerName
),
RankedCustomers AS (
    SELECT
        Country,
        CustomerName,
        TotalSpent,
        RANK() OVER (
            PARTITION BY Country
            ORDER BY TotalSpent DESC
        ) AS SpendingRank
    FROM CustomerSpending
)
SELECT Country, CustomerName, TotalSpent
FROM RankedCustomers
WHERE SpendingRank = 1
ORDER BY Country;


""", conn)

,Country,CustomerName,TotalSpent
0,Argentina,Diego Gutiérrez,37.62
1,Australia,Mark Taylor,37.62
2,Austria,Astrid Gruber,42.62
3,Belgium,Daan Peeters,37.62
4,Brazil,Luís Gonçalves,39.62
5,Canada,François Tremblay,39.62
6,Chile,Luis Rojas,46.62
7,Czech Republic,Helena Holý,49.62
8,Denmark,Kara Nielsen,37.62
9,Finland,Terhi Hämäläinen,41.62


**Q87.** Compute a 3-invoice moving average of invoice totals per customer (ordered by date).

In [95]:
pd.read_sql("""
SELECT
    CustomerId,
    InvoiceId,
    InvoiceDate,
    Total,
    AVG(Total) OVER (
        PARTITION BY CustomerId
        ORDER BY InvoiceDate, InvoiceId
        ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
    ) AS ThreeInvoiceMovingAvg
FROM Invoice
ORDER BY CustomerId, InvoiceDate;

""", conn)

,CustomerId,InvoiceId,InvoiceDate,Total,ThreeInvoiceMovingAvg
0,1,98,2022-03-11 00:00:00,3.98,3.980000
1,1,121,2022-06-13 00:00:00,3.96,3.970000
2,1,143,2022-09-15 00:00:00,5.94,4.626667
3,1,195,2023-05-06 00:00:00,0.99,3.630000
4,1,316,2024-10-27 00:00:00,1.98,2.970000
...,...,...,...,...,...
407,59,45,2021-07-08 00:00:00,5.94,4.950000
408,59,97,2022-02-26 00:00:00,1.99,3.963333
409,59,218,2023-08-20 00:00:00,1.98,3.303333
410,59,229,2023-09-30 00:00:00,13.86,5.943333


**Q88.** For each genre, show each track's length and its percentile rank within the genre using PERCENT_RANK().

In [96]:
# pd.read_sql("""

# """, conn)

**Q89.** Use NTILE(4) to bucket customers into spending quartiles. Show quartile, name, and total spent.

In [97]:
# pd.read_sql("""

# """, conn)

**Q90.** Show monthly revenue and the difference from the previous month using LAG() over InvoiceDate.

In [98]:
# pd.read_sql("""

# """, conn)

**Q91.** Using DENSE_RANK(), find which invoice is each customer's highest-value purchase (rank 1).

In [99]:
# pd.read_sql("""

# """, conn)

**Q92.** Show cumulative revenue per billing country ordered by InvoiceDate using SUM() OVER ().

In [100]:
# pd.read_sql("""

# """, conn)

**Q93.** For each track purchased, show how many days passed since that customer's previous purchase (using LAG + date math).

In [101]:
# pd.read_sql("""

# """, conn)

**Q94.** Rank genres by total revenue per year. Show year, genre, revenue, and rank within that year.

In [102]:
# pd.read_sql("""

# """, conn)

**Q95.** Using a CTE + window function, flag each customer invoice as above or below that customer's own average.

In [103]:
# pd.read_sql("""

# """, conn)

**Q96.** Show each invoice with its percentage contribution to that customer's total lifetime spend.

In [104]:
# pd.read_sql("""

# """, conn)

**Q97.** Using ROW_NUMBER(), identify each customer's first-ever purchase (date and amount).

In [105]:
# pd.read_sql("""

# """, conn)

**Q98.** Compute a running count of unique customers acquired over time (ordered by their first purchase date).

In [106]:
# pd.read_sql("""

# """, conn)

**Q99.** Using a CTE and window functions, find any customer whose spending in a given year was more than double their prior year spending.

In [107]:
# pd.read_sql("""

# """, conn)

**Q100.** Build a full customer RFM table: Recency (days since last purchase), Frequency (number of invoices), Monetary (total spent). Rank each dimension with NTILE(4) and compute a composite RFM score.

In [108]:
# pd.read_sql("""

# """, conn)

In [109]:
conn.close()